In [13]:
%run graph.py

......
----------------------------------------------------------------------
Ran 6 tests in 0.005s

OK


In [24]:
# 6.0002 Problem Set 5
# Graph optimization
# Name:
# Collaborators:
# Time:

#
# Finding shortest paths through MIT buildings
#
import unittest
from graph import Digraph, Node, WeightedEdge

#
# Problem 2: Building up the Campus Map
#
# Problem 2a: Designing your graph
#
# What do the graph's nodes represent in this problem? What
# do the graph's edges represent? Where are the distances
# represented?
#
# Answer:
# The nodes represent each building at MIT. The graph edges are a 4-tuple consisting of the two building nodes, 
#the distance from the 1st building to the 2nd building, and lastly the value how much of that distance is outdoor.
# Because of this, distances are represented as weights of the graph edges.


# Problem 2b: Implementing load_map
def load_map(map_filename):
    """
    Parses the map file and constructs a directed graph

    Parameters:
        map_filename : name of the map file

    Assumes:
        Each entry in the map file consists of the following four positive
        integers, separated by a blank space:
            From To TotalDistance DistanceOutdoors
        e.g.
            32 76 54 23
        This entry would become an edge from 32 to 76.

    Returns:
        a Digraph representing the map
    """
    print("Loading map from file...")
    map_graph = Digraph()
    with open(map_filename, 'r') as file:
        
        #get values for each entry
        line = file.readline()
        line_split = line.split()
        
        while len(line) != 0: 
            
            #add nodes first
            node1 = Node(line_split[0])
            node2 = Node(line_split[1])
            for node in (node1, node2):
                try:
                    map_graph.add_node(node)
                except ValueError:
                    pass
            
            #add edge
            weight_edge = WeightedEdge(node1,
                                       node2, 
                                       int(line_split[-2]),
                                       int(line_split[-1])
                                      )                
            map_graph.add_edge(weight_edge)
            line = file.readline() #get next line
            line_split = line.split()
    
    print('Loading Done!')
    return map_graph
            
# Problem 2c: Testing load_map
# Include the lines used to test load_map below, but comment them out

#test_map = load_map('test_load_map.txt')
#print(test_map)

#
# Problem 3: Finding the Shorest Path using Optimized Search Method
#
# Problem 3a: Objective function
#


# What is the objective function for this problem? What are the constraints?
#
# Answer: The objective function is the sum of the distance weights of the path taken. In other words, the total distance
#         travelled on the path taken.
#         The constraint would be first the total distance cannot exceed the maximum distance outdoors,
#         and we look at optimising given fixed first and last nodes. 

# Problem 3b: Implement get_best_path
def get_best_path(digraph, start, end, path, max_dist_outdoors, best_dist,
                  best_path):
    """
    Finds the shortest path between buildings subject to constraints.

    Parameters:
        digraph: Digraph instance
            The graph on which to carry out the search
        start: string
            Building number at which to start
        end: string
            Building number at which to end
        path: list composed of [[list of strings], int, int]
            Represents the current path of nodes being traversed. Contains
            a list of node names, total distance traveled, and total
            distance outdoors.
        max_dist_outdoors: int
            Maximum distance spent outdoors on a path
        best_dist: int
            The smallest distance between the original start and end node
            for the initial problem that you are trying to solve
        best_path: list of strings
            The shortest path found so far between the original start
            and end node.

    Returns:
        A tuple with the shortest-path from start to end, represented by
        a list of building numbers (in strings), [n_1, n_2, ..., n_k],
        where there exists an edge from n_i to n_(i+1) in digraph,
        for all 1 <= i < k and the distance of that path.

        If there exists no path that satisfies max_total_dist and
        max_dist_outdoors constraints, then return None.
    """
    start_node = Node(start)
    end_node = Node(end)
    path = path + [start]

    #get distances of current path
    dist_outdoors = 0
    dist = 0
    for j in range(len(path)-1):
                path_edges = digraph.get_edges_for_node(Node(path[j]))
                for path_edge in path_edges:
                        if path_edge.get_destination() == Node(path[j+1]):
                            dist_outdoors += path_edge.get_outdoor_distance()
                            dist += path_edge.get_total_distance()

    #outdoor constraint
    if dist_outdoors > max_dist_outdoors:
        return (best_path, best_dist)

    #check if start and end nodes are in the digraph
    if (digraph.has_node(start_node) == False or 
        digraph.has_node(end_node) == False):
        raise TypeError('The start and end should be in the Digraph')
    #trivial case
    elif start_node == end_node:
            return (path, dist)
    
    else:
        #get edges in a start node
        start_node_edges = digraph.get_edges_for_node(start_node)

        #loop over each child node/edge
        for start_edge in start_node_edges:
            child_node = start_edge.get_destination()
            child = child_node.get_name()
            # avoid cycles, check node not in path
            if child not in path:
                #do not exceed current optimum path
                if (best_path == None) or dist < best_dist:
                        new_path, new_distance = get_best_path(digraph, child, end, path, 
                                                            max_dist_outdoors, best_dist, best_path)
                        #finding new path, ensure distance still below constraint
                        if new_path != None and new_distance < best_dist:
                            best_path = new_path
                            best_dist = new_distance
        
    return (best_path, best_dist)


# Problem 3c: Implement directed_dfs
def directed_dfs(digraph, start, end, max_total_dist, max_dist_outdoors):
    """
    Finds the shortest path from start to end using a directed depth-first
    search. The total distance traveled on the path must not
    exceed max_total_dist, and the distance spent outdoors on this path must
    not exceed max_dist_outdoors.

    Parameters:
        digraph: Digraph instance
            The graph on which to carry out the search
        start: string
            Building number at which to start
        end: string
            Building number at which to end
        max_total_dist: int
            Maximum total distance on a path
        max_dist_outdoors: int
            Maximum distance spent outdoors on a path

    Returns:
        The shortest-path from start to end, represented by
        a list of building numbers (in strings), [n_1, n_2, ..., n_k],
        where there exists an edge from n_i to n_(i+1) in digraph,
        for all 1 <= i < k

        If there exists no path that satisfies max_total_dist and
        max_dist_outdoors constraints, then raises a ValueError.
    """
    optimum_path, optimum_dist = get_best_path(digraph, start, end, [], 
                                               max_dist_outdoors, 
                                               float('inf'),
                                               None)
    if optimum_dist <= max_total_dist:
         return optimum_path
    else:
         raise ValueError('No path exists which fits given constraints')
     


# ================================================================
# Begin tests -- you do not need to modify anything below this line
# ================================================================

class Ps2Test(unittest.TestCase):
    LARGE_DIST = 99999

    def setUp(self):
        self.graph = load_map("mit_map.txt")

    def test_load_map_basic(self):
        self.assertTrue(isinstance(self.graph, Digraph))
        self.assertEqual(len(self.graph.nodes), 37)
        all_edges = []
        for _, edges in self.graph.edges.items():
            all_edges += edges  # edges must be dict of node -> list of edges
        all_edges = set(all_edges)
        self.assertEqual(len(all_edges), 129)

    def _print_path_description(self, start, end, total_dist, outdoor_dist):
        constraint = ""
        if outdoor_dist != Ps2Test.LARGE_DIST:
            constraint = "without walking more than {}m outdoors".format(
                outdoor_dist)
        if total_dist != Ps2Test.LARGE_DIST:
            if constraint:
                constraint += ' or {}m total'.format(total_dist)
            else:
                constraint = "without walking more than {}m total".format(
                    total_dist)

        print("------------------------")
        print("Shortest path from Building {} to {} {}".format(
            start, end, constraint))

    def _test_path(self,
                   expectedPath,
                   total_dist=LARGE_DIST,
                   outdoor_dist=LARGE_DIST):
        start, end = expectedPath[0], expectedPath[-1]
        self._print_path_description(start, end, total_dist, outdoor_dist)
        dfsPath = directed_dfs(self.graph, start, end, total_dist, outdoor_dist)
        print("Expected: ", expectedPath)
        print("DFS: ", dfsPath)
        self.assertEqual(expectedPath, dfsPath)

    def _test_impossible_path(self,
                              start,
                              end,
                              total_dist=LARGE_DIST,
                              outdoor_dist=LARGE_DIST):
        self._print_path_description(start, end, total_dist, outdoor_dist)
        with self.assertRaises(ValueError):
            directed_dfs(self.graph, start, end, total_dist, outdoor_dist)

    def test_path_one_step(self):
        self._test_path(expectedPath=['32', '56'])

    def test_path_no_outdoors(self):
        self._test_path(
            expectedPath=['32', '36', '26', '16', '56'], outdoor_dist=0)

    def test_path_multi_step(self):
        self._test_path(expectedPath=['2', '3', '7', '9'])

    def test_path_multi_step_no_outdoors(self):
        self._test_path(
            expectedPath=['2', '4', '10', '13', '9'], outdoor_dist=0)

    def test_path_multi_step2(self):
        self._test_path(expectedPath=['1', '4', '12', '32'])

    def test_path_multi_step_no_outdoors2(self):
        self._test_path(
            expectedPath=['1', '3', '10', '4', '12', '24', '34', '36', '32'],
            outdoor_dist=0)

    def test_impossible_path1(self):
        self._test_impossible_path('8', '50', outdoor_dist=0)

    def test_impossible_path2(self):
        self._test_impossible_path('10', '32', total_dist=100)


if __name__ == "__main__":
    unittest.main(argv=[''], exit=False)  # ignore Jupyter's kernel args


...

............
----------------------------------------------------------------------
Ran 15 tests in 0.072s

OK


Loading map from file...
Loading Done!
------------------------
Shortest path from Building 8 to 50 without walking more than 0m outdoors
Loading map from file...
Loading Done!
------------------------
Shortest path from Building 10 to 32 without walking more than 100m total
Loading map from file...
Loading Done!
Loading map from file...
Loading Done!
------------------------
Shortest path from Building 2 to 9 
Expected:  ['2', '3', '7', '9']
DFS:  ['2', '3', '7', '9']
Loading map from file...
Loading Done!
------------------------
Shortest path from Building 1 to 32 
Expected:  ['1', '4', '12', '32']
DFS:  ['1', '4', '12', '32']
Loading map from file...
Loading Done!
------------------------
Shortest path from Building 2 to 9 without walking more than 0m outdoors
Expected:  ['2', '4', '10', '13', '9']
DFS:  ['2', '4', '10', '13', '9']
Loading map from file...
Loading Done!
------------------------
Shortest path from Building 1 to 32 without walking more than 0m outdoors
Expected:  ['1

In [5]:
test_map = load_map('test_load_map.txt')
print(test_map)

Loading map from file...
Loading Done!
a->b (10, 9)
a->c (12, 2)
b->c (1, 1)


In [4]:
aa = TestGraph()
bb = aa.setUp()

In [18]:
ggg  = Digraph()
xx = Node('a'); yy = Node('b'); zz = Node('c')
ggg.add_node(xx); ggg.add_node(yy); ggg.add_node(zz)

xxx = WeightedEdge(xx, yy, 15, 10)
yyy = WeightedEdge(xx, zz, 14, 6)
zzz = WeightedEdge(yy, zz, 3,  1)
ggg.add_edge(xxx); ggg.add_edge(yyy); ggg.add_edge(zzz)

print(type(ggg.get_edges_for_node(xx)))
print()
#edges have src and destination
for edges in (ggg.get_edges_for_node(xx)):
    print(edges.get_destination())
    print(type(edges.get_destination()))
# a->b (15, 10)
# a->c (14, 6)
# b->c (3, 1)

<class 'list'>

b
<class '__main__.Node'>
c
<class '__main__.Node'>


In [10]:
float('inf')

inf